In [ ]:
import math
import pandas as pd
from scipy.stats import norm

# =========================
# DATA FROM THE ASSIGNMENT
# =========================

# SKU A (EOQ)
DA = 48_000
KA = 240
hA = 1.20

# SKU B ((Q,R))
muB = 220        # mean daily demand
sigmaB = 60      # standard deviation of daily demand
L = 4            # lead time (days)
DB_annual = 80_000
KB = 180
hB = 2.50
CSL = 0.95       # Cycle Service Level

# SKU C (Newsvendor)
cC = 18
pC = 39
vC = 6
muC = 4800       # mean period demand
sigmaC = 1400    # standard deviation of period demand

# =========================
# SKU A – EOQ
# =========================

EOQ_A = math.sqrt(2 * DA * KA / hA)                 # Q* = sqrt(2DK/h)
orders_A = DA / EOQ_A                               # orders/year = D/Q
cost_A = (DA / EOQ_A) * KA + (EOQ_A / 2) * hA       # ordering + holding

results_A = pd.DataFrame([{
    "EOQ_A": EOQ_A,
    "Orders_per_year_A": orders_A,
    "Annual_cost_A(ordering+holding)": cost_A
}])

# Sensitivity ±20% in DA and hA
levels = [-0.2, 0.2]
sens_A_EOQ = pd.DataFrame(
    index=["hA -20%", "hA +20%"],
    columns=["DA -20%", "DA +20%"],
    dtype=float
)

for lv_h in levels:
    for lv_D in levels:
        DA_test = DA * (1 + lv_D)
        hA_test = hA * (1 + lv_h)
        EOQ_test = math.sqrt(2 * DA_test * KA / hA_test)

        row = "hA -20%" if lv_h == -0.2 else "hA +20%"
        col = "DA -20%" if lv_D == -0.2 else "DA +20%"
        sens_A_EOQ.loc[row, col] = EOQ_test

# =========================
# SKU B – (Q,R) Model
# =========================

EOQ_B = math.sqrt(2 * DB_annual * KB / hB)          # EOQ

mu_LT = muB * L                                     # mean demand during lead time
sigma_LT = sigmaB * math.sqrt(L)                    # standard deviation during lead time

zB = float(norm.ppf(CSL))                           # z for CSL
SS_B = zB * sigma_LT                                # safety stock
R_B = mu_LT + SS_B                                  # reorder point

ordering_cost_B = (DB_annual / EOQ_B) * KB          # ordering cost
holding_cost_B = (EOQ_B / 2 + SS_B) * hB            # holding cost (incl. SS)
total_cost_B = ordering_cost_B + holding_cost_B

results_B_policy = pd.DataFrame([{
    "EOQ_B": EOQ_B,
    "mu_LT": mu_LT,
    "sigma_LT": sigma_LT,
    "CSL": CSL,
    "z(CSL)": zB,
    "SafetyStock_SS": SS_B,
    "ReorderPoint_R": R_B
}])

results_B_costs = pd.DataFrame([{
    "OrderingCost_B": ordering_cost_B,
    "HoldingCost_B(incl_SS)": holding_cost_B,
    "TotalAnnualCost_B": total_cost_B
}])

# =========================
# SKU C – Newsvendor
# =========================

Cu = pC - cC                                         # underage cost
Co = cC - vC                                         # overage cost
CR = Cu / (Cu + Co)                                  # critical ratio

zC = float(norm.ppf(CR))                             # z(CR)
Q_star = muC + zC * sigmaC                           # Q* = mu + zsigma

results_C = pd.DataFrame([{
    "Cu(p-c)": Cu,
    "Co(c-v)": Co,
    "CriticalRatio": CR,
    "z(CR)": zC,
    "Q_star": Q_star
}])

# Sensitivity in sigma (0.8sigma, 1.0sigma, 1.2sigma)
factors = [0.8, 1.0, 1.2]
sens_C = pd.DataFrame([
    {"sigma_factor": f, "sigma_used": sigmaC * f, "Q_star": muC + zC * (sigmaC * f)}
    for f in factors
])


print("✅ SKU A – EOQ (Results)")
display(results_A)

print("\n✅ SKU A – EOQ Sensitivity (±20% in DA and hA)")
display(sens_A_EOQ)

print("\n✅ SKU B – (Q,R) Policy (EOQ, R, SS)")
display(results_B_policy)

print("\n✅ SKU B – Annual Costs")
display(results_B_costs)

print("\n✅ SKU C – Newsvendor (Results)")
display(results_C)

print("\n✅ SKU C – Q* Sensitivity with respect to sigma")
display(sens_C)
